# TraceLens — Experiment Notebook
End-to-end pipeline: parse → align → score → rank → LLM → evaluate

Run cells sequentially. **Set your Groq API key in cell 1 before running LLM cells.**

In [ ]:
import os, sys, json
sys.path.insert(0, '..')  # adjust if running from notebooks/

os.environ['GROQ_API_KEY'] = ''  # <-- paste your key here

import yaml
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config loaded. Sources:', list(cfg['data']['sources'].keys()))

## 1. Parse & Inspect a Single Trace Pair

In [ ]:
from src.trace_parser import parse_trace
import pprint

RAW = cfg['data']['raw_base']
SOURCE = 'efe_irem'
TRACE_ID = 'wikipedia'

pass_path = f"{RAW}/efe- irem traces/wikipedia/wikipedia_correct.json"
fail_path = f"{RAW}/efe- irem traces/wikipedia/wikipedia_incorrect.json"

passing = parse_trace(SOURCE, pass_path)
failing = parse_trace(SOURCE, fail_path)
print(f'Pass: {len(passing)} steps | Fail: {len(failing)} steps')
print('\nSample step (fail step 13):')
pprint.pprint(failing[13])

## 2. Align + Score

In [ ]:
from src.trace_aligner import align
from src.anomaly_detector import AnomalyDetector

aligned = align(passing, failing)
detector = AnomalyDetector(weights=cfg['weights'])
scored = detector.compute_scores(aligned)

top10 = sorted(scored, key=lambda x: x['combined_score'], reverse=True)[:10]
for s in top10:
    print(f'Step {s["step_id"]:3d} | score={s["combined_score"]:.3f} '
          f'net={s["network_score"]:.2f} con={s["console_score"]:.2f} '
          f'act={s["action_score"]:.2f} int={s["intent_score"]:.2f} '
          f'| {s["action"][:55]}')

## 3. Rank + LLM Re-rank

In [ ]:
from src.ranker import Ranker
from src.llm_reasoner import LlmReasoner

ranker = Ranker(top_k=cfg['ranking']['top_k'], pre_llm_k=cfg['ranking']['pre_llm_k'])
candidates = ranker.candidates_for_llm(scored)
heuristic_top5 = ranker.rank_heuristic(scored)

print('Heuristic top-5:')
for s in heuristic_top5:
    print(f'  Step {s["step_id"]:3d} score={s["combined_score"]:.3f} | {s["action"][:60]}')

if os.environ.get('GROQ_API_KEY'):
    llm = LlmReasoner(model=cfg['model']['llm_model'], temperature=cfg['model']['temperature'])
    reranked_ids = llm.rerank(candidates)
    final_ranked = ranker.apply_llm_reranking(reranked_ids, scored)
    final_ranked = ranker.add_rank_metadata(final_ranked)
    print('\nLLM re-ranked top-5:')
    for s in final_ranked:
        print(f'  #{s["rank"]} Step {s["step_id"]:3d} | {s["action"][:60]}')
else:
    final_ranked = ranker.add_rank_metadata(heuristic_top5)
    print('(No API key — using heuristic ranking)')

## 4. LLM Diagnosis + Stakeholder Summary

In [ ]:
if os.environ.get('GROQ_API_KEY'):
    diagnosis = llm.diagnose(final_ranked)
    summary = llm.stakeholder_summary(diagnosis)
    print('TECHNICAL DIAGNOSIS')
    print(json.dumps(diagnosis, indent=2))
    print('\nSTAKEHOLDER SUMMARY')
    print(summary)
else:
    print('Set GROQ_API_KEY to run LLM diagnosis.')

## 5. Evaluate Against Ground Truth

In [ ]:
from src.evaluation import Evaluator

with open(cfg['data']['ground_truth']) as f:
    gt = json.load(f)

actual_fault = gt[SOURCE][TRACE_ID]['fault_step']
print(f'Ground truth fault step: {actual_fault}')

evaluator = Evaluator(top_k=5)
result = evaluator.evaluate_trace(final_ranked, actual_fault)

print(f'Hit@1={result["hit@1"]}  Hit@3={result["hit@3"]}  Hit@5={result["hit@5"]}')
print(f'Rank position: {result["rank_position"]}')
print(f'Rank distance: {result["rank_distance"]}')
print(f'Top-1 step distance: {result["top1_step_distance"]}')
print(f'MAD@5: {result["mad@5"]}')
print('\nPer-rank step distances:')
for d in result['step_distances']:
    marker = ' <- ACTUAL' if d['step_id'] == actual_fault else ''
    print(f'  Rank {d["rank"]}: Step {d["step_id"]} | step_dist={d["step_distance"]}{marker}')

## 6. Run Full Pipeline on All 22 Traces
Run from the project root terminal, or uncomment below.

In [ ]:
# Uncomment to run:
# !python main.py --no-llm                              # heuristic only (fast)
# !python main.py                                       # with LLM (needs GROQ_API_KEY)
# !python main.py --source efe_irem --trace gutenberg   # single trace

## 7. Aggregate Results

In [ ]:
import pathlib
agg_path = pathlib.Path('../outputs/metrics/aggregate.json')
if agg_path.exists():
    with open(agg_path) as f:
        agg = json.load(f)
    print('Aggregate metrics:')
    for k, v in agg.items():
        print(f'  {k}: {v}')
else:
    print('Run the full pipeline first (section 6 above).')